In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
# Setup
import os, sys, json, random, csv, zipfile
import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import (efficientnet_b0, EfficientNet_B0_Weights,
                                 vit_b_16, ViT_B_16_Weights)
from PIL import Image
from sklearn.metrics import (accuracy_score, f1_score,
                              confusion_matrix, cohen_kappa_score)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

os.system('pip install -q timm pytorch-ignite')
import timm
from ignite.engine import Engine, Events
from ignite.metrics import Accuracy, Loss, RunningAverage
from ignite.handlers import EarlyStopping, ModelCheckpoint

# Clone repo (baseline checkpoint / shared modules live here)
os.chdir('/kaggle/working')
if not os.path.exists('fyp-food-classification'):
    os.system('git clone https://github.com/Ahmad-techs/fyp-food-classification.git')
else:
    os.system('cd fyp-food-classification && git pull')
sys.path.insert(0, '/kaggle/working/fyp-food-classification/src')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

def set_seeds(seed=42):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seeds(42)

# Update this to match your Kaggle input path — check with: !ls /kaggle/input/
FOOD11_ROOT = '/kaggle/input/datasets/trolukovich/food11-image-dataset'

OUT_DIR  = '/kaggle/working/results_food11_v3'
CKPT_DIR = '/kaggle/working/checkpoints_food11_v3'
os.makedirs(OUT_DIR,  exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

print('Setup complete')

Cloning into 'fyp-food-classification'...


Device: cuda
GPU: Tesla T4
Setup complete


In [2]:
# Coarse mapping + Food-11 Dataset + DataLoaders
CLASS_NAMES = [
    'Bread', 'Dairy product', 'Dessert', 'Egg', 'Fried food',
    'Meat', 'Noodles-Pasta', 'Rice', 'Seafood', 'Soup', 'Vegetable-Fruit'
]

# Group 0: Carbohydrate | 1: Protein | 2: Dessert/High-sugar | 3: Other/Light
COARSE_MAPPING = {
    'Bread': 0, 'Noodles-Pasta': 0, 'Rice': 0,
    'Meat': 1, 'Egg': 1, 'Seafood': 1, 'Dairy product': 1,
    'Dessert': 2,
    'Fried food': 3, 'Soup': 3, 'Vegetable-Fruit': 3
}

COARSE_NAMES = {
    0: 'Carbohydrate',
    1: 'Protein',
    2: 'Dessert / High-sugar',
    3: 'Other / Light'
}

def get_coarse_label(fine_name: str) -> int:
    return COARSE_MAPPING.get(fine_name, 3)

print('Coarse mapping:')
for cls in CLASS_NAMES:
    gid = get_coarse_label(cls)
    print(f'  {cls:20s} -> Group {gid} ({COARSE_NAMES[gid]})')

# Numeric folder -> class name mapping (Food-11 numbered 0-10)
NUMERIC_TO_NAME = {
    '0': 'Bread', '1': 'Dairy product', '2': 'Dessert',
    '3': 'Egg',   '4': 'Fried food',    '5': 'Meat',
    '6': 'Noodles-Pasta', '7': 'Rice',  '8': 'Seafood',
    '9': 'Soup',  '10': 'Vegetable-Fruit'
}

def get_transforms(split: str):
    if split == 'training':
        return transforms.Compose([
            transforms.Resize(256),
            transforms.RandomCrop(224),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.2, contrast=0.2,
                                   saturation=0.2, hue=0.05),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
        ])
    return transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])


class Food11Dataset(Dataset):
    """Food-11 dataset. Handles named (Bread/Meat/...) or numeric (0/1/...)
    class folders. Returns (image, fine_label, coarse_label)."""

    def __init__(self, root_dir: str, split: str):
        self.transform = get_transforms(split)
        self.samples   = []

        split_dir = os.path.join(root_dir, split)
        folders   = sorted([d for d in os.listdir(split_dir)
                            if os.path.isdir(os.path.join(split_dir, d))])

        for folder in folders:
            if folder in NUMERIC_TO_NAME:
                class_name = NUMERIC_TO_NAME[folder]
                fine_id    = int(folder)
            elif folder in CLASS_NAMES:
                class_name = folder
                fine_id    = CLASS_NAMES.index(folder)
            else:
                print(f"  WARNING: unknown folder '{folder}' — skipping")
                continue

            coarse_id   = get_coarse_label(class_name)
            folder_path = os.path.join(split_dir, folder)

            for fname in os.listdir(folder_path):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.samples.append((
                        os.path.join(folder_path, fname), fine_id, coarse_id
                    ))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, fine, coarse = self.samples[idx]
        try:
            img = Image.open(path).convert('RGB')
            return self.transform(img), fine, coarse
        except Exception:
            return self.__getitem__((idx + 1) % len(self.samples))


print('Building datasets...')
train_ds = Food11Dataset(FOOD11_ROOT, 'training')
val_ds   = Food11Dataset(FOOD11_ROOT, 'validation')
test_ds  = Food11Dataset(FOOD11_ROOT, 'evaluation')

print(f'  Train:      {len(train_ds)} images')
print(f'  Validation: {len(val_ds)} images')
print(f'  Test:       {len(test_ds)} images')

BS = 32
train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BS, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BS, shuffle=False, num_workers=2, pin_memory=True)

imgs, fl, cl = next(iter(train_loader))
print(f'\nBatch shapes: imgs={imgs.shape} fine={fl.shape} coarse={cl.shape}')
print(f'Fine labels range:   [{fl.min().item()}, {fl.max().item()}]  (expected 0-10)')
print(f'Coarse labels range: [{cl.min().item()}, {cl.max().item()}]  (expected 0-3)')
print('DataLoaders ready')


Coarse mapping:
  Bread                -> Group 0 (Carbohydrate)
  Dairy product        -> Group 1 (Protein)
  Dessert              -> Group 2 (Dessert / High-sugar)
  Egg                  -> Group 1 (Protein)
  Fried food           -> Group 3 (Other / Light)
  Meat                 -> Group 1 (Protein)
  Noodles-Pasta        -> Group 0 (Carbohydrate)
  Rice                 -> Group 0 (Carbohydrate)
  Seafood              -> Group 1 (Protein)
  Soup                 -> Group 3 (Other / Light)
  Vegetable-Fruit      -> Group 3 (Other / Light)
Building datasets...
  Train:      9866 images
  Validation: 3430 images
  Test:       3347 images

Batch shapes: imgs=torch.Size([32, 3, 224, 224]) fine=torch.Size([32]) coarse=torch.Size([32])
Fine labels range:   [0, 10]  (expected 0-10)
Coarse labels range: [0, 3]  (expected 0-3)
DataLoaders ready


In [3]:
# Three Model Definitions (dual head: 11 fine, 4 coarse)
class EfficientNetDualHead(nn.Module):
    def __init__(self, num_fine=11, num_coarse=4, dropout=0.3):
        super().__init__()
        b = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        self.features    = b.features
        self.avgpool     = b.avgpool
        self.dropout     = nn.Dropout(dropout)
        self.fine_head   = nn.Linear(1280, num_fine)
        self.coarse_head = nn.Linear(1280, num_coarse)

    def forward(self, x):
        x = self.features(x); x = self.avgpool(x)
        x = torch.flatten(x, 1); x = self.dropout(x)
        return self.fine_head(x), self.coarse_head(x)

    def freeze_backbone(self):
        for p in self.features.parameters(): p.requires_grad = False
    def unfreeze_top(self, n=3):
        for block in list(self.features.children())[-n:]:
            for p in block.parameters(): p.requires_grad = True
    def unfreeze_all(self):
        for p in self.features.parameters(): p.requires_grad = True


class ViTDualHead(nn.Module):
    def __init__(self, num_fine=11, num_coarse=4, dropout=0.3):
        super().__init__()
        vit = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
        vit.heads     = nn.Identity()
        self.backbone = vit
        self.dropout  = nn.Dropout(dropout)
        self.fine_head   = nn.Linear(768, num_fine)
        self.coarse_head = nn.Linear(768, num_coarse)

    def forward(self, x):
        x = self.backbone(x); x = self.dropout(x)
        return self.fine_head(x), self.coarse_head(x)

    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = False
    def unfreeze_top(self, n=3):
        layers = list(self.backbone.encoder.layers.children())
        for layer in layers[-n:]:
            for p in layer.parameters(): p.requires_grad = True
        for p in self.backbone.encoder.ln.parameters(): p.requires_grad = True
    def unfreeze_all(self):
        for p in self.backbone.parameters(): p.requires_grad = True


class CoAtNetDualHead(nn.Module):
    def __init__(self, num_fine=11, num_coarse=4, dropout=0.3):
        super().__init__()
        self.backbone    = timm.create_model('coatnet_0_rw_224', pretrained=True, num_classes=0)
        feat_dim         = self.backbone.num_features
        self.dropout     = nn.Dropout(dropout)
        self.fine_head   = nn.Linear(feat_dim, num_fine)
        self.coarse_head = nn.Linear(feat_dim, num_coarse)

    def forward(self, x):
        x = self.backbone(x); x = self.dropout(x)
        return self.fine_head(x), self.coarse_head(x)

    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = False
    def unfreeze_top(self, n=3):
        for name, mod in list(self.backbone.named_children())[-n:]:
            for p in mod.parameters(): p.requires_grad = True
    def unfreeze_all(self):
        for p in self.backbone.parameters(): p.requires_grad = True


print('Testing model shapes (num_fine=11, num_coarse=4) ...')
dummy = torch.zeros(2, 3, 224, 224)
with torch.no_grad():
    for Cls, name in [(EfficientNetDualHead, 'EfficientNet-B0'),
                       (ViTDualHead,          'ViT-B/16'),
                       (CoAtNetDualHead,      'CoAtNet-0')]:
        m = Cls()
        f, c = m(dummy)
        p = sum(x.numel() for x in m.parameters()) / 1e6
        assert f.shape == (2, 11) and c.shape == (2, 4)
        print(f'  {name:15s}: fine={f.shape}  coarse={c.shape}  params={p:.1f}M ✓')
        del m
print('All models verified')

Testing model shapes (num_fine=11, num_coarse=4) ...
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 116MB/s] 


  EfficientNet-B0: fine=torch.Size([2, 11])  coarse=torch.Size([2, 4])  params=4.0M ✓
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:02<00:00, 173MB/s]  


  ViT-B/16       : fine=torch.Size([2, 11])  coarse=torch.Size([2, 4])  params=85.8M ✓


model.safetensors:   0%|          | 0.00/110M [00:00<?, ?B/s]

  CoAtNet-0      : fine=torch.Size([2, 11])  coarse=torch.Size([2, 4])  params=26.7M ✓
All models verified


In [6]:
# %% Cell 4 — Ignite-based Training (Phase 1: 10 epochs | Phase 2: EarlyStopping patience=5)
LAM = 0.35  # coarse loss weight, matches Food-5k pipeline

def make_loss_fn(lam):
    crit = nn.CrossEntropyLoss()
    def loss_fn(fine_out, coarse_out, fl, cl):
        if lam > 0:
            return (1 - lam) * crit(fine_out, fl) + lam * crit(coarse_out, cl)
        return crit(fine_out, fl)
    return loss_fn


def build_engines(model, lr, lam):
    """Builds an ignite Engine for training and one for validation.
    Both attach running/epoch metrics: 'loss', 'fine_acc', 'coarse_acc'."""
    loss_fn = make_loss_fn(lam)
    optimizer = AdamW([p for p in model.parameters() if p.requires_grad],
                       lr=lr, weight_decay=1e-4)

    def train_step(engine, batch):
        model.train()
        imgs, fl, cl = batch
        imgs, fl, cl = imgs.to(device), fl.to(device), cl.to(device)
        optimizer.zero_grad()
        fine_out, coarse_out = model(imgs)
        loss = loss_fn(fine_out, coarse_out, fl, cl)
        loss.backward()
        optimizer.step()
        return {'loss': loss.item()}

    def eval_step(engine, batch):
        model.eval()
        with torch.no_grad():
            imgs, fl, cl = batch
            imgs, fl, cl = imgs.to(device), fl.to(device), cl.to(device)
            fine_out, coarse_out = model(imgs)
            loss = loss_fn(fine_out, coarse_out, fl, cl)
        return {'loss': loss.item(), 'fine_pred': fine_out, 'fine_y': fl,
                'coarse_pred': coarse_out, 'coarse_y': cl}

    trainer   = Engine(train_step)
    evaluator = Engine(eval_step)

    RunningAverage(output_transform=lambda o: o['loss']).attach(trainer, 'loss')
    Accuracy(output_transform=lambda o: (o['fine_pred'],   o['fine_y'])).attach(evaluator, 'fine_acc')
    Accuracy(output_transform=lambda o: (o['coarse_pred'], o['coarse_y'])).attach(evaluator, 'coarse_acc')
    Loss(nn.CrossEntropyLoss(), output_transform=lambda o: (o['fine_pred'], o['fine_y'])).attach(evaluator, 'val_loss')

    return trainer, evaluator, optimizer


def run_phase(model, phase_tag, max_epochs, lr, lam, save_path,
              history, use_early_stopping, patience=5):
    """
    One training phase driven by ignite.
    Phase 1 (heads only):   use_early_stopping=False, runs exactly max_epochs.
    Phase 2+ (fine-tuning): use_early_stopping=True, EarlyStopping handler
                            (patience=5) monitors validation loss (minimize).
    """
    set_seeds(42)
    trainer, evaluator, _ = build_engines(model, lr=lr, lam=lam)
    best = {'fine_acc': history.get('best_fine_acc', 0.0)}

    @trainer.on(Events.EPOCH_COMPLETED)
    def _run_validation(engine):
        evaluator.run(val_loader)
        m = evaluator.state.metrics
        fa, ca, vl = m['fine_acc'] * 100, m['coarse_acc'] * 100, m['val_loss']
        avg_loss = trainer.state.metrics.get('loss', float('nan'))
        print(f'  {phase_tag} Ep{engine.state.epoch:03d}/{max_epochs}  '
              f'train_loss={avg_loss:.4f}  val_loss={vl:.4f}  '
              f'Fine={fa:.2f}%  Coarse={ca:.2f}%')

        history.setdefault('val_fine',   []).append(fa)
        history.setdefault('val_coarse', []).append(ca)
        history.setdefault('val_loss',   []).append(vl)
        history.setdefault('loss',       []).append(avg_loss)

        if fa > best['fine_acc']:
            best['fine_acc'] = fa
            history['best_fine_acc'] = fa
            torch.save({
                'model_state_dict': model.state_dict(),
                'best_fine_acc':    fa,
                'lambda':           lam,
                'epoch':            engine.state.epoch,
                'phase':            phase_tag,
            }, save_path)
            print(f'  ✓ Checkpoint saved (best Fine={fa:.2f}%)')

    if use_early_stopping:
        # ignite EarlyStopping: score_function must return higher-is-better,
        # so we negate validation loss (professor: monitor validation loss).
        def score_function(engine):
            return -engine.state.metrics['val_loss']

        es_handler = EarlyStopping(patience=patience,
                                    score_function=score_function,
                                    trainer=trainer)
        evaluator.add_event_handler(Events.COMPLETED, es_handler)
        print(f'  EarlyStopping attached: patience={patience}, monitors val_loss (min)')

    trainer.run(train_loader, max_epochs=max_epochs)
    return history


def train_model_v3(ModelClass, model_name, save_path, lam=LAM,
                    phase2_max_epochs=100):
    """
      Phase 1: classification heads only, exactly 10 epochs, no early stop.
      Phase 2: fine-tune (top-3 unfrozen), EarlyStopping patience=5,
               capped at phase2_max_epochs as a safety ceiling.
    """
    print(f'\n{"="*65}')
    print(f'  {model_name}  |  lambda={lam}')
    print(f'  Schedule: P1=10ep (heads, no ES) | P2<= {phase2_max_epochs}ep + EarlyStopping(patience=5)')
    print(f'{"="*65}')

    model = ModelClass(num_fine=11, num_coarse=4).to(device)
    h = {}

    print('\n-- Phase 1: Heads only | 10 epochs | lr=1e-3 | no early stop --')
    model.freeze_backbone()
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'   Trainable params: {trainable:,}')
    h = run_phase(model, 'P1', max_epochs=10, lr=1e-3, lam=lam,
                  save_path=save_path, history=h, use_early_stopping=False)

    print('\n-- Phase 2: Top-3 unfrozen | EarlyStopping patience=5 | lr=1e-4 --')
    model.unfreeze_top(n=3)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'   Trainable params: {trainable:,}')
    h = run_phase(model, 'P2', max_epochs=phase2_max_epochs, lr=1e-4, lam=lam,
                  save_path=save_path, history=h, use_early_stopping=True, patience=5)

    print(f'\n✓ {model_name} done.')
    print(f'  Total epochs run: {len(h["val_fine"])}')
    print(f'  Best val Fine Top-1: {h["best_fine_acc"]:.2f}%')
    return h

print('Ignite training functions ready')

Ignite training functions ready


In [7]:
# Train EfficientNet-B0
h_eff = train_model_v3(EfficientNetDualHead, 'EfficientNet-B0',
                        f'{CKPT_DIR}/food11_v3_effnet.pth')
print(f'\n EfficientNet-B0: {h_eff["best_fine_acc"]:.2f}% ({len(h_eff["val_fine"])} epochs)')
with open(f'{OUT_DIR}/effnet_history.json', 'w') as f:
    json.dump({k: v for k, v in h_eff.items() if isinstance(v, list)}, f)


  EfficientNet-B0  |  lambda=0.35
  Schedule: P1=10ep (heads, no ES) | P2<= 100ep + EarlyStopping(patience=5)

-- Phase 1: Heads only | 10 epochs | lr=1e-3 | no early stop --
   Trainable params: 19,215
  P1 Ep001/10  train_loss=0.9433  val_loss=0.8353  Fine=75.83%  Coarse=75.71%
  ✓ Checkpoint saved (best Fine=75.83%)
  P1 Ep002/10  train_loss=0.8402  val_loss=0.7252  Fine=78.45%  Coarse=77.17%
  ✓ Checkpoint saved (best Fine=78.45%)
  P1 Ep003/10  train_loss=0.7860  val_loss=0.6677  Fine=79.71%  Coarse=77.67%
  ✓ Checkpoint saved (best Fine=79.71%)
  P1 Ep004/10  train_loss=0.7849  val_loss=0.6628  Fine=79.74%  Coarse=77.90%
  ✓ Checkpoint saved (best Fine=79.74%)
  P1 Ep005/10  train_loss=0.7420  val_loss=0.6493  Fine=79.85%  Coarse=78.57%
  ✓ Checkpoint saved (best Fine=79.85%)
  P1 Ep006/10  train_loss=0.7583  val_loss=0.6303  Fine=80.35%  Coarse=78.48%
  ✓ Checkpoint saved (best Fine=80.35%)
  P1 Ep007/10  train_loss=0.7443  val_loss=0.6160  Fine=80.58%  Coarse=79.01%
  ✓ Checkp

2026-07-22 19:18:42,131 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


  P2 Ep014/100  train_loss=0.0800  val_loss=0.2998  Fine=92.39%  Coarse=92.97%
  ✓ Checkpoint saved (best Fine=92.39%)

✓ EfficientNet-B0 done.
  Total epochs run: 24
  Best val Fine Top-1: 92.39%

 EfficientNet-B0: 92.39% (24 epochs)


In [9]:
# Train ViT-B/16
h_vit = train_model_v3(ViTDualHead, 'ViT-B/16',
                        f'{CKPT_DIR}/food11_v3_vit.pth')
print(f'\n ViT-B/16: {h_vit["best_fine_acc"]:.2f}% ({len(h_vit["val_fine"])} epochs)')
with open(f'{OUT_DIR}/vit_history.json', 'w') as f:
    json.dump({k: v for k, v in h_vit.items() if isinstance(v, list)}, f)


  ViT-B/16  |  lambda=0.35
  Schedule: P1=10ep (heads, no ES) | P2<= 100ep + EarlyStopping(patience=5)

-- Phase 1: Heads only | 10 epochs | lr=1e-3 | no early stop --
   Trainable params: 11,535
  P1 Ep001/10  train_loss=0.4805  val_loss=0.4383  Fine=86.97%  Coarse=86.03%
  ✓ Checkpoint saved (best Fine=86.97%)
  P1 Ep002/10  train_loss=0.4094  val_loss=0.3797  Fine=88.22%  Coarse=87.64%
  ✓ Checkpoint saved (best Fine=88.22%)
  P1 Ep003/10  train_loss=0.3760  val_loss=0.3580  Fine=88.89%  Coarse=87.70%
  ✓ Checkpoint saved (best Fine=88.89%)
  P1 Ep004/10  train_loss=0.3609  val_loss=0.3374  Fine=89.36%  Coarse=88.25%
  ✓ Checkpoint saved (best Fine=89.36%)
  P1 Ep005/10  train_loss=0.3562  val_loss=0.3282  Fine=89.42%  Coarse=88.31%
  ✓ Checkpoint saved (best Fine=89.42%)
  P1 Ep006/10  train_loss=0.3348  val_loss=0.3193  Fine=89.97%  Coarse=88.78%
  ✓ Checkpoint saved (best Fine=89.97%)
  P1 Ep007/10  train_loss=0.3572  val_loss=0.3254  Fine=89.94%  Coarse=88.43%
  P1 Ep008/10  tr

2026-07-22 20:35:43,406 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


  P2 Ep007/100  train_loss=0.0342  val_loss=0.2748  Fine=93.70%  Coarse=93.79%

✓ ViT-B/16 done.
  Total epochs run: 17
  Best val Fine Top-1: 93.82%

 ViT-B/16: 93.82% (17 epochs)


In [10]:
# Train CoAtNet-0
h_coat = train_model_v3(CoAtNetDualHead, 'CoAtNet-0',
                         f'{CKPT_DIR}/food11_v3_coatnet.pth')
print(f'\n CoAtNet-0: {h_coat["best_fine_acc"]:.2f}% ({len(h_coat["val_fine"])} epochs)')
with open(f'{OUT_DIR}/coatnet_history.json', 'w') as f:
    json.dump({k: v for k, v in h_coat.items() if isinstance(v, list)}, f)

histories = {'EfficientNet-B0': h_eff, 'ViT-B/16': h_vit, 'CoAtNet-0': h_coat}
print('\n' + '='*50)
print('FOOD-11 V3 VAL SUMMARY')
print('='*50)
for name, h in histories.items():
    print(f'  {name:18s}: {h["best_fine_acc"]:.2f}%  ({len(h["val_fine"])} epochs)')


  CoAtNet-0  |  lambda=0.35
  Schedule: P1=10ep (heads, no ES) | P2<= 100ep + EarlyStopping(patience=5)

-- Phase 1: Heads only | 10 epochs | lr=1e-3 | no early stop --
   Trainable params: 11,535
  P1 Ep001/10  train_loss=0.5810  val_loss=0.5045  Fine=84.81%  Coarse=84.87%
  ✓ Checkpoint saved (best Fine=84.81%)
  P1 Ep002/10  train_loss=0.4848  val_loss=0.4426  Fine=86.38%  Coarse=86.01%
  ✓ Checkpoint saved (best Fine=86.38%)
  P1 Ep003/10  train_loss=0.4440  val_loss=0.4155  Fine=86.94%  Coarse=86.50%
  ✓ Checkpoint saved (best Fine=86.94%)
  P1 Ep004/10  train_loss=0.4629  val_loss=0.3941  Fine=87.26%  Coarse=87.26%
  ✓ Checkpoint saved (best Fine=87.26%)
  P1 Ep005/10  train_loss=0.4510  val_loss=0.3898  Fine=87.38%  Coarse=86.88%
  ✓ Checkpoint saved (best Fine=87.38%)
  P1 Ep006/10  train_loss=0.4302  val_loss=0.3795  Fine=87.64%  Coarse=87.20%
  ✓ Checkpoint saved (best Fine=87.64%)
  P1 Ep007/10  train_loss=0.4477  val_loss=0.3759  Fine=87.90%  Coarse=87.26%
  ✓ Checkpoint s

2026-07-22 21:24:24,474 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


  P2 Ep010/100  train_loss=0.0916  val_loss=0.3059  Fine=92.13%  Coarse=93.91%

✓ CoAtNet-0 done.
  Total epochs run: 20
  Best val Fine Top-1: 94.17%

 CoAtNet-0: 94.17% (20 epochs)

FOOD-11 V3 VAL SUMMARY
  EfficientNet-B0   : 92.39%  (24 epochs)
  ViT-B/16          : 93.82%  (17 epochs)
  CoAtNet-0         : 94.17%  (20 epochs)


In [11]:
# Evaluate on Test Set + Confusion Matrices (fine + coarse)
def full_evaluate(model, loader, class_names, coarse_names, label, out_dir):
    model.eval()
    all_ft, all_fp, all_ct, all_cp = [], [], [], []
    with torch.no_grad():
        for imgs, fl, cl in loader:
            fine_out, coarse_out = model(imgs.to(device))
            all_fp.extend(fine_out.argmax(1).cpu().numpy())
            all_ft.extend(fl.numpy())
            all_cp.extend(coarse_out.argmax(1).cpu().numpy())
            all_ct.extend(cl.numpy())

    all_ft, all_fp = np.array(all_ft), np.array(all_fp)
    all_ct, all_cp = np.array(all_ct), np.array(all_cp)

    acc   = accuracy_score(all_ft, all_fp) * 100
    f1    = f1_score(all_ft, all_fp, average='macro', zero_division=0) * 100
    kappa = cohen_kappa_score(all_ft, all_fp)
    coarse_acc = accuracy_score(all_ct, all_cp) * 100

    print(f'\n{"="*50}\n  {label}  [TEST SET]\n{"="*50}')
    print(f'  Fine Top-1 Accuracy:   {acc:.2f}%')
    print(f'  Fine Macro F1:         {f1:.2f}%')
    print(f'  Fine Cohen Kappa:      {kappa:.4f}')
    print(f'  Coarse Top-1 Accuracy: {coarse_acc:.2f}%')

    cm = confusion_matrix(all_ft, all_fp, labels=list(range(len(class_names))))
    print('\n  Per-class recall (fine):')
    for i, name in enumerate(class_names):
        if cm[i].sum() > 0:
            print(f'    {name:18s}: {cm[i, i] / cm[i].sum() * 100:.1f}%')

    cm_pct = cm.astype(float) / np.clip(cm.sum(axis=1, keepdims=True), 1, None) * 100
    fig, ax = plt.subplots(figsize=(9, 8))
    sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=ax, linewidths=0.5, cbar_kws={'label': 'Recall (%)'})
    ax.set_xlabel('Predicted Class'); ax.set_ylabel('True Class')
    ax.set_title(f'Fine-grained Confusion Matrix — {label}', fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    safe = label.replace(' ', '_').replace('/', '_')
    fig_path = f'{out_dir}/{safe}_fine_cm.png'
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f'  Fine confusion matrix saved: {fig_path}')

    cm_c = confusion_matrix(all_ct, all_cp, labels=list(coarse_names.keys()))
    cm_c_pct = cm_c.astype(float) / np.clip(cm_c.sum(axis=1, keepdims=True), 1, None) * 100
    fig, ax = plt.subplots(figsize=(6, 5))
    labels_c = [coarse_names[i] for i in sorted(coarse_names)]
    sns.heatmap(cm_c_pct, annot=True, fmt='.1f', cmap='Oranges',
                xticklabels=labels_c, yticklabels=labels_c,
                ax=ax, linewidths=0.5, cbar_kws={'label': 'Recall (%)'})
    ax.set_xlabel('Predicted Group'); ax.set_ylabel('True Group')
    ax.set_title(f'Coarse Confusion Matrix — {label}', fontweight='bold')
    plt.xticks(rotation=20, ha='right')
    plt.tight_layout()
    coarse_path = f'{out_dir}/{safe}_coarse_cm.png'
    plt.savefig(coarse_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f'  Coarse confusion matrix saved: {coarse_path}')

    return {'condition': label, 'top1': acc, 'f1': f1, 'kappa': kappa,
            'coarse_acc': coarse_acc, 'cm_fine': cm, 'cm_coarse': cm_c}


experiments = [
    ('EfficientNet-B0', EfficientNetDualHead, f'{CKPT_DIR}/food11_v3_effnet.pth'),
    ('ViT-B/16',        ViTDualHead,          f'{CKPT_DIR}/food11_v3_vit.pth'),
    ('CoAtNet-0',       CoAtNetDualHead,      f'{CKPT_DIR}/food11_v3_coatnet.pth'),
]

results = []
for label, ModelClass, ckpt_path in experiments:
    ckpt  = torch.load(ckpt_path, map_location=device, weights_only=False)
    model = ModelClass(num_fine=11, num_coarse=4).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    r = full_evaluate(model, test_loader, CLASS_NAMES, COARSE_NAMES,
                       f'{label} Food-11', OUT_DIR)
    results.append(r)

print('\n' + '='*65)
print('FOOD-11 FINAL TEST RESULTS')
print('='*65)
print(f'{"Model":20s} {"Fine Top-1":>12} {"F1":>8} {"Kappa":>8} {"Coarse Top-1":>14}')
print('-'*65)
for r in results:
    print(f'{r["condition"]:20s} {r["top1"]:>11.2f}% {r["f1"]:>7.2f}% '
          f'{r["kappa"]:>8.4f} {r["coarse_acc"]:>13.2f}%')


  EfficientNet-B0 Food-11  [TEST SET]
  Fine Top-1 Accuracy:   93.49%
  Fine Macro F1:         93.98%
  Fine Cohen Kappa:      0.9269
  Coarse Top-1 Accuracy: 93.31%

  Per-class recall (fine):
    Bread             : 92.7%
    Dairy product     : 86.5%
    Dessert           : 89.4%
    Egg               : 91.0%
    Fried food        : 90.9%
    Meat              : 93.5%
    Noodles-Pasta     : 98.0%
    Rice              : 100.0%
    Seafood           : 94.4%
    Soup              : 98.0%
    Vegetable-Fruit   : 98.3%
  Fine confusion matrix saved: /kaggle/working/results_food11_v3/EfficientNet-B0_Food-11_fine_cm.png
  Coarse confusion matrix saved: /kaggle/working/results_food11_v3/EfficientNet-B0_Food-11_coarse_cm.png

  ViT-B/16 Food-11  [TEST SET]
  Fine Top-1 Accuracy:   95.07%
  Fine Macro F1:         95.27%
  Fine Cohen Kappa:      0.9447
  Coarse Top-1 Accuracy: 95.31%

  Per-class recall (fine):
    Bread             : 93.8%
    Dairy product     : 91.9%
    Dessert         

In [12]:
# Epoch Count Summary (for paper methodology)
print('EPOCH COUNT PER MODEL (for paper methodology section)')
print('='*55)
print(f'{"Model":20s} {"P1":>6} {"P2":>6} {"Total":>8}')
print('-'*55)
p1_fixed = 10
epoch_records = {}
for name, h in histories.items():
    total  = len(h['val_fine'])
    p2_run = max(0, total - p1_fixed)
    epoch_records[name] = {'P1': p1_fixed, 'P2': p2_run, 'Total': total}
    print(f'{name:20s} {p1_fixed:>6} {p2_run:>6} {total:>8}')

print('\nNote for paper:')
print('  Phase 1: 10 epochs (heads only, no early stopping)')
print('  Phase 2: EarlyStopping (ignite handler), patience=5, monitors val_loss')
print(f'  lambda (coarse loss weight): {LAM}')
print('  Optimizer: AdamW (weight decay=1e-4)')

with open(f'{OUT_DIR}/epoch_counts.json', 'w') as f:
    json.dump(epoch_records, f, indent=2)
print(f'\nEpoch counts saved to epoch_counts.json')

EPOCH COUNT PER MODEL (for paper methodology section)
Model                    P1     P2    Total
-------------------------------------------------------
EfficientNet-B0          10     14       24
ViT-B/16                 10      7       17
CoAtNet-0                10     10       20

Note for paper:
  Phase 1: 10 epochs (heads only, no early stopping)
  Phase 2: EarlyStopping (ignite handler), patience=5, monitors val_loss
  lambda (coarse loss weight): 0.35
  Optimizer: AdamW (weight decay=1e-4)

Epoch counts saved to epoch_counts.json


In [13]:
# Training Curves + Comparison Chart
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Food-11 (v3) — Training Curves\nPhase 1: 10ep | Phase 2: EarlyStopping(patience=5)',
             fontsize=12, fontweight='bold')
colours = ['#c0392b', '#2980b9', '#8e44ad']

for (name, h), col in zip(histories.items(), colours):
    axes[0].plot(h['val_loss'],  label=name, color=col)
    axes[1].plot(h['val_fine'],  label=name, color=col)

for ax, title, ylabel in [(axes[0], 'Validation Loss', 'Loss'),
                          (axes[1], 'Val Fine Top-1 (%)', 'Accuracy (%)')]:
    ax.set_title(title, fontsize=11); ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    ax.legend(fontsize=9)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/food11_v3_training_curves.png', dpi=300, bbox_inches='tight')
plt.close()
print('Training curves saved.')

fig, ax = plt.subplots(figsize=(8, 5))
models = [r['condition'] for r in results]
accs   = [r['top1']      for r in results]
bars   = ax.bar(models, accs, color=colours, edgecolor='white', width=0.5)
ax.set_title('Food-11 Test Accuracy — 3 Models (Updated Schedule)', fontsize=12, fontweight='bold')
ax.set_ylabel('Top-1 Accuracy (%)')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
for bar, v in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{v:.2f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/food11_v3_comparison.png', dpi=300, bbox_inches='tight')
plt.close()
print('Comparison chart saved.')

Training curves saved.
Comparison chart saved.


In [14]:
# Save CSV + Download Zip
csv_path = f'{OUT_DIR}/FOOD11_V3_RESULTS.csv'
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['condition', 'top1', 'f1', 'kappa', 'coarse_acc'])
    writer.writeheader()
    for r in results:
        writer.writerow({'condition': r['condition'],
                          'top1': round(r['top1'], 2),
                          'f1': round(r['f1'], 2),
                          'kappa': round(r['kappa'], 4),
                          'coarse_acc': round(r['coarse_acc'], 2)})
print(f'CSV saved: {csv_path}')

zip_path = '/kaggle/working/food11_v3_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in [OUT_DIR, CKPT_DIR]:
        for root, dirs, files in os.walk(folder):
            for file in files:
                fp   = os.path.join(root, file)
                name = os.path.relpath(fp, '/kaggle/working')
                zf.write(fp, name)
print('Zip ready.')

from IPython.display import FileLink, display
display(FileLink('food11_v3_results.zip'))

CSV saved: /kaggle/working/results_food11_v3/FOOD11_V3_RESULTS.csv
Zip ready.


/kaggle/working/food11_v3_results.zip

In [17]:
import math
import pandas as pd

ERR_DIR = f'{OUT_DIR}/error_analysis'
os.makedirs(ERR_DIR, exist_ok=True)

def run_inference(ModelClass, ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model = ModelClass(num_fine=11, num_coarse=4).to(device)
    model.load_state_dict(ckpt['model_state_dict']); model.eval()
    rows = []
    with torch.no_grad():
        for imgs, fl, cl in test_loader:
            fo, _ = model(imgs.to(device))
            probs = torch.softmax(fo, dim=1)
            preds = fo.argmax(1).cpu().numpy()
            confs = probs.max(1).values.cpu().numpy()
            for t, p, c in zip(fl.numpy(), preds, confs):
                rows.append({'true_label': int(t), 'pred_label': int(p),
                             'true_class': CLASS_NAMES[t], 'pred_class': CLASS_NAMES[p],
                             'confidence': round(float(c), 4)})
    return pd.DataFrame(rows)

summary_rows = []
all_confused = {}

for label, ModelClass, ckpt_path in experiments:
    model_name = label
    df = run_inference(ModelClass, ckpt_path)
    safe = model_name.replace(' ', '_').replace('/', '_')
    df.to_csv(f'{ERR_DIR}/{safe}_test_predictions.csv', index=False)

    mis = df[df['true_label'] != df['pred_label']]
    print(f'\n{model_name}: {len(mis)}/{len(df)} misclassified ({100*len(mis)/len(df):.2f}%)')

    recall_rows = []
    for i, name in enumerate(CLASS_NAMES):
        cls_df = df[df['true_label'] == i]
        if len(cls_df) == 0: continue
        recall = (cls_df['pred_label'] == i).mean() * 100
        recall_rows.append({'class': name, 'recall_pct': round(recall, 2), 'n_samples': len(cls_df)})
    recall_df = pd.DataFrame(recall_rows).sort_values('recall_pct')
    recall_df.to_csv(f'{ERR_DIR}/{safe}_per_class_recall.csv', index=False)
    print('  Worst-performing classes:'); print(recall_df.head(3).to_string(index=False))

    pair_counts = mis.groupby(['true_class', 'pred_class']).size().reset_index(name='count').sort_values('count', ascending=False)
    pair_counts.to_csv(f'{ERR_DIR}/{safe}_confused_pairs.csv', index=False)
    print('  Top confused pairs:'); print(pair_counts.head(5).to_string(index=False))

    all_confused[model_name] = (pair_counts, mis, safe)
    summary_rows.append({'model': model_name, 'total': len(df), 'misclassified': len(mis),
                          'error_rate_pct': round(100*len(mis)/len(df), 2)})

pd.DataFrame(summary_rows).to_csv(f'{ERR_DIR}/error_analysis_summary.csv', index=False)
print(f'\nSummary saved: {ERR_DIR}/error_analysis_summary.csv')


EfficientNet-B0: 218/3347 misclassified (6.51%)
  Worst-performing classes:
        class  recall_pct  n_samples
Dairy product       86.49        148
      Dessert       89.40        500
   Fried food       90.94        287
  Top confused pairs:
   true_class pred_class  count
          Egg      Bread     16
Dairy product    Dessert     12
          Egg    Dessert     10
        Bread        Egg     10
         Meat Fried food     10

ViT-B/16: 165/3347 misclassified (4.93%)
  Worst-performing classes:
        class  recall_pct  n_samples
          Egg       88.06        335
Dairy product       91.89        148
   Fried food       93.73        287
  Top confused pairs:
   true_class pred_class  count
          Egg      Bread     20
Dairy product    Dessert      8
          Egg    Dessert      8
      Dessert       Soup      7
        Bread    Dessert      7

CoAtNet-0: 158/3347 misclassified (4.72%)
  Worst-performing classes:
        class  recall_pct  n_samples
Dairy product       9

In [18]:
for model_name, (pair_counts, mis, safe) in all_confused.items():
    if len(pair_counts) == 0:
        print(f'{model_name}: no misclassifications.'); continue
    top_true, top_pred = pair_counts.iloc[0]['true_class'], pair_counts.iloc[0]['pred_class']
    errs = mis[(mis['true_class'] == top_true) & (mis['pred_class'] == top_pred)].sort_values('confidence', ascending=False)

    # Recover filenames by walking the evaluation folder for the true class
    folder_candidates = [str(CLASS_NAMES.index(top_true)), top_true]
    folder = next((f for f in folder_candidates if os.path.isdir(os.path.join(FOOD11_ROOT, 'evaluation', f))), None)
    files_in_folder = os.listdir(os.path.join(FOOD11_ROOT, 'evaluation', folder)) if folder else []

    rows_n = min(len(errs), 24); ncols = min(6, rows_n); nrows = math.ceil(rows_n / max(ncols, 1)) if rows_n else 0
    if rows_n == 0:
        continue
    fig, axes = plt.subplots(nrows, ncols, figsize=(3*ncols, 3*nrows))
    axes = np.array(axes).reshape(-1)
    for i in range(nrows * ncols):
        ax = axes[i]; ax.axis('off')
        if i < rows_n and i < len(files_in_folder):
            fname = files_in_folder[i]
            img_path = os.path.join(FOOD11_ROOT, 'evaluation', folder, fname)
            if os.path.exists(img_path):
                ax.imshow(Image.open(img_path).convert('RGB'))
            ax.set_title(fname, fontsize=7)
    fig.suptitle(f'{model_name}: "{top_true}" misclassified as "{top_pred}" (sample images from {top_true})',
                 fontsize=11, fontweight='bold')
    plt.tight_layout()
    out_path = f'{ERR_DIR}/{safe}_top_confused_pair_contact_sheet.png'
    plt.savefig(out_path, dpi=200, bbox_inches='tight'); plt.close()
    print(f'{model_name}: contact sheet saved -> {out_path}')

print('\nOpen each *_top_confused_pair_contact_sheet.png and *_confused_pairs.csv')
print('to document the visual traits driving each confusion for the results section.')

EfficientNet-B0: contact sheet saved -> /kaggle/working/results_food11_v3/error_analysis/EfficientNet-B0_top_confused_pair_contact_sheet.png
ViT-B/16: contact sheet saved -> /kaggle/working/results_food11_v3/error_analysis/ViT-B_16_top_confused_pair_contact_sheet.png
CoAtNet-0: contact sheet saved -> /kaggle/working/results_food11_v3/error_analysis/CoAtNet-0_top_confused_pair_contact_sheet.png

Open each *_top_confused_pair_contact_sheet.png and *_confused_pairs.csv
to document the visual traits driving each confusion for the results section.


In [19]:
zip_path = '/kaggle/working/food11_v3_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in [OUT_DIR, CKPT_DIR]:
        for root, dirs, files in os.walk(folder):
            for file in files:
                fp = os.path.join(root, file)
                zf.write(fp, os.path.relpath(fp, '/kaggle/working'))
print('Zip re-created with error analysis included.')

from IPython.display import FileLink, display
display(FileLink('food11_v3_results.zip'))


Zip re-created with error analysis included.


/kaggle/working/food11_v3_results.zip